# Cross-Industry Accelerator — Load Warehouse
### Auto-Load Tables into Fabric Data Warehouse with DDL Generation

This notebook loads **all discovered tables** into the Fabric Data Warehouse,
dynamically generating `CREATE TABLE` DDL from inferred Spark schemas.

**What it does:**
1. Reads industry configuration from `00_Industry_Config`
2. For each table (dim, fact-batch, fact-event, stream):
   - Infers schema from CSV
   - Generates SQL `CREATE TABLE IF NOT EXISTS` DDL
   - Loads data via `synapsesql()` connector
3. Generates a load summary with row counts and status

> **Prerequisites:**
> 1. Run `01_Data_Ingestion.ipynb` to validate sources
> 2. A Fabric **Data Warehouse** must exist (name configured in `00_Industry_Config`)
> 3. The notebook must have connectivity to the warehouse endpoint

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════
%run ./00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# TYPE MAPPING & DDL GENERATION UTILITIES (ZT-hardened)
# ════════════════════════════════════════════════════════════════════════

from pyspark.sql.types import (
    StringType, IntegerType, LongType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, DecimalType, ShortType, ByteType
)
import pandas as pd

# Spark type → Warehouse SQL type mapping
SPARK_TO_SQL = {
    StringType:    "NVARCHAR(4000)",
    IntegerType:   "INT",
    LongType:      "BIGINT",
    ShortType:     "SMALLINT",
    ByteType:      "TINYINT",
    FloatType:     "FLOAT",
    DoubleType:    "FLOAT",
    BooleanType:   "BIT",
    DateType:      "DATE",
    TimestampType: "DATETIME2",
}

def spark_type_to_sql(spark_type):
    """Convert a Spark DataType instance to SQL Server type string."""
    if isinstance(spark_type, DecimalType):
        return f"DECIMAL({spark_type.precision}, {spark_type.scale})"
    return SPARK_TO_SQL.get(type(spark_type), "NVARCHAR(4000)")

def generate_ddl(table_name, schema, warehouse_schema="dbo"):
    """Generate CREATE TABLE IF NOT EXISTS DDL from Spark schema.
    ZT: All identifiers are validated before interpolation into SQL."""
    # ZT: Validate table name and schema name
    sanitize_identifier(table_name)
    sanitize_identifier(warehouse_schema)

    # ZT: Validate and build column definitions with sanitized names
    col_defs = []
    for field in schema.fields:
        sanitize_identifier(field.name)  # Raises ValueError if unsafe
        null_clause = "" if field.nullable else " NOT NULL"
        col_defs.append(f"[{field.name}] {spark_type_to_sql(field.dataType)}{null_clause}")

    columns = ",\n    ".join(col_defs)
    return f"""IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = '{table_name}')
CREATE TABLE [{warehouse_schema}].[{table_name}] (
    {columns}
)"""

print("✅ DDL generation utilities loaded (ZT: column name sanitization enabled).")
print(f"   Warehouse target: {WAREHOUSE_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD ALL TABLES → FABRIC DATA WAREHOUSE (ZT-hardened)
# ════════════════════════════════════════════════════════════════════════

ALL_TABLES = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT + STREAM_TABLES
warehouse_schema = "dbo"

results = []

print(f"Loading {len(ALL_TABLES)} tables into Warehouse ({WAREHOUSE_NAME}.{warehouse_schema})...\n")

for table_name in ALL_TABLES:
    csv_path = f"{_FS_CSV_PATH}/{table_name}.csv"

    # Determine category for reporting
    if table_name in DIM_TABLES:
        category = "Dimension"
    elif table_name in FACT_TABLES_BATCH:
        category = "Fact (Batch)"
    elif table_name in FACT_TABLES_EVENT:
        category = "Fact (Event)"
    else:
        category = "Streaming"

    try:
        # Read CSV with schema inference
        df = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(csv_path))

        row_count = df.count()

        # Generate and execute DDL (ZT: column names validated inside generate_ddl)
        ddl = generate_ddl(table_name, df.schema, warehouse_schema)
        spark.sql(ddl)

        # Write data via synapsesql connector
        full_table = f"{WAREHOUSE_NAME}.{warehouse_schema}.{table_name}"
        df.write.mode("append").synapsesql(full_table)

        results.append((table_name, category, row_count, len(df.columns), "✓"))
        log_audit_event("WAREHOUSE_LOAD", full_table, f"{row_count} rows written")
        print(f"  ✓ {table_name:<45} {row_count:>6} rows  {len(df.columns):>3} cols")

    except ValueError as ve:
        # ZT: Identifier validation failure
        results.append((table_name, category, 0, 0, f"✗ ZT: {ve}"))
        log_audit_event("WAREHOUSE_LOAD", table_name, str(ve), "BLOCKED")
        print(f"  ✗ {table_name:<45} ZT BLOCKED: {ve}")
    except Exception as e:
        results.append((table_name, category, 0, 0, f"✗ {e}"))
        log_audit_event("WAREHOUSE_LOAD", table_name, str(e), "ERROR")
        print(f"  ✗ {table_name:<45} ERROR: {e}")

ok_count = sum(1 for r in results if r[4] == '✓')
print(f"\n✅ Tables loaded: {ok_count}/{len(ALL_TABLES)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# GENERATED DDL PREVIEW
# ════════════════════════════════════════════════════════════════════════
# Re-read one table to show the DDL that was generated (for verification)

if ALL_TABLES:
    sample_table = ALL_TABLES[0]
    sample_path = f"{_FS_CSV_PATH}/{sample_table}.csv"
    sample_df = (spark.read
                 .option("header", True)
                 .option("inferSchema", True)
                 .csv(sample_path))
    print(f"Sample DDL for '{sample_table}':\n")
    print(generate_ddl(sample_table, sample_df.schema, warehouse_schema))

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# WAREHOUSE LOAD SUMMARY
# ════════════════════════════════════════════════════════════════════════

summary_df = pd.DataFrame(results, columns=["Table", "Category", "Rows", "Columns", "Status"])

print(f"\n{'═'*75}")
print(f"WAREHOUSE LOAD SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*75}")
print(summary_df.to_string(index=False))
print(f"\n{'─'*75}")

for cat in ["Dimension", "Fact (Batch)", "Fact (Event)", "Streaming"]:
    cat_df = summary_df[summary_df['Category'] == cat]
    if not cat_df.empty:
        ok = (cat_df['Status'] == '✓').sum()
        total_rows = cat_df[cat_df['Status'] == '✓']['Rows'].sum()
        print(f"  {cat:<16} {ok}/{len(cat_df)} tables  {total_rows:>8,} rows")

print(f"\n  TOTAL: {summary_df[summary_df['Status']=='✓']['Rows'].sum():,} rows loaded")
print(f"\n✅ Warehouse loading complete.")
print(f"   Next: Run 04_Create_Semantic_Model.ipynb to create the Power BI semantic model.")